## 准备数据

In [4]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

def mnist_dataset(batch_size=128):
    tr = transforms.ToTensor()
    train_ds = datasets.MNIST(root='./mnist/', train=True, transform=tr, download=True)
    test_ds = datasets.MNIST(root='./mnist/', train=False, transform=tr, download=True)
    return DataLoader(train_ds, batch_size=batch_size, shuffle=True), DataLoader(test_ds, batch_size=256, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


## 建立模型

In [5]:
class myConvModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Linear(7 * 7 * 64, 10)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        logits = self.classifier(x)
        probs = torch.softmax(logits, dim=-1)
        return logits, probs

model = myConvModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


## 编译， fit以及evaluate

In [6]:
train_ds, test_ds = mnist_dataset()
criterion = nn.CrossEntropyLoss()
for epoch in range(5):
    model.train()
    for x, y in train_ds:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits, _ = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in test_ds:
            x, y = x.to(device), y.to(device)
            logits, _ = model(x)
            correct += (logits.argmax(dim=1) == y).sum().item()
            total += y.size(0)
    print('epoch', epoch+1, ': test acc', correct / total)


epoch 1 : test acc 0.983
epoch 2 : test acc 0.9864
epoch 3 : test acc 0.99
epoch 4 : test acc 0.9908
epoch 5 : test acc 0.9927
